In [36]:

# CELL 1 — OptiCore dependency setup and environment verification

import sys
import subprocess
import importlib.util
import importlib.metadata

# Map import names to their pip package names.
REQUIRED_PACKAGES = {
    "ultralytics": "ultralytics",
    "fastapi": "fastapi",
    "uvicorn": "uvicorn[standard]",
    "pyngrok": "pyngrok",
    "cv2": "opencv-python-headless",
    "websockets": "websockets",
}

missing_packages = [
    package
    for module, package in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        *missing_packages
    ])
else:
    print("All required packages are already installed.")

# Import only after any necessary installation.
import torch
import cv2
import numpy as np
import fastapi
import uvicorn
import ultralytics
import pyngrok
import websockets

print("\n" + "=" * 45)
print("OPTICORE ENVIRONMENT CHECK")
print("=" * 45)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)
print("Ultralytics:", ultralytics.__version__)
print("OpenCV:", cv2.__version__)
print("FastAPI:", fastapi.__version__)
print("Uvicorn:", uvicorn.__version__)
print("pyngrok:", importlib.metadata.version("pyngrok"))
print("WebSockets:", websockets.__version__)

if not torch.cuda.is_available():
    raise RuntimeError(
        "Kaggle GPU is unavailable. Enable GPU in Notebook Settings, "
        "then rerun this cell."
    )

print("\nCell 1 completed successfully.")


All required packages are already installed.

OPTICORE ENVIRONMENT CHECK
Python: 3.13.15
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Ultralytics: 8.4.194
OpenCV: 4.14.0
FastAPI: 0.141.1
Uvicorn: 0.52.4
pyngrok: 9.1.0
WebSockets: 15.0.1

Cell 1 completed successfully.


In [37]:

# CELL 2 — OptiCore model, GPU optimization, tracking state and priorities

import os
import time
import threading
import numpy as np
import torch

from collections import Counter, defaultdict, deque
from dataclasses import dataclass, field
from ultralytics import YOLO

# --------------------------------------------------
# 1. Model configuration
# --------------------------------------------------

MODEL_PATH = (
    "/kaggle/input/models/"
    "dharshan8197/opticore/pytorch/default/1/best.pt"
)

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(
        f"OptiCore model not found:\n{MODEL_PATH}\n"
        "Verify the Kaggle model input attachment."
    )

DEVICE = 0 if torch.cuda.is_available() else "cpu"

IMGSZ = 640
CONFIDENCE = 0.25
MAX_FRAME_WIDTH = 640
JPEG_QUALITY = 65

# Use FP16 on supported CUDA hardware to reduce inference cost.
USE_HALF = bool(torch.cuda.is_available())

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends.cudnn, "allow_tf32"):
        torch.backends.cudnn.allow_tf32 = True

# --------------------------------------------------
# 2. Load model once
# --------------------------------------------------

print("Loading OptiCore model...")
model = YOLO(MODEL_PATH)
CLASS_NAMES = model.names

if not CLASS_NAMES:
    raise RuntimeError("The model loaded without class names.")

print("Model loaded.")
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("FP16 inference:", USE_HALF)
print("Image size:", IMGSZ)
print("Number of classes:", len(CLASS_NAMES))

# Warm up using the same precision setting intended for inference.
dummy_frame = np.zeros((480, 640, 3), dtype=np.uint8)

warmup_start = time.perf_counter()
with torch.inference_mode():
    model.predict(
        source=dummy_frame,
        imgsz=IMGSZ,
        conf=CONFIDENCE,
        device=DEVICE,
        half=USE_HALF,
        verbose=False,
    )

print(
    "Warm-up completed in",
    round(time.perf_counter() - warmup_start, 2),
    "seconds"
)

# --------------------------------------------------
# 3. Synchronization and metrics
# --------------------------------------------------

INFERENCE_LOCK = threading.Lock()
METRICS_LOCK = threading.Lock()
TRACK_HISTORY_LOCK = threading.Lock()
ALERT_HISTORY_LOCK = threading.Lock()
PRODUCER_LOCK = threading.Lock()

CURRENT_COUNTS = Counter()
UNIQUE_TRACKS = defaultdict(set)

METRICS = {
    "frames_processed": 0,
    "last_inference_ms": 0.0,
    "avg_inference_ms": 0.0,
    "fps_ema": 0.0,
    "frames_rejected": 0,
    "last_inference_unix": None,
}

# --------------------------------------------------
# 4. Per-track temporal history
# --------------------------------------------------

@dataclass
class TrackHistory:
    samples: deque = field(default_factory=lambda: deque(maxlen=8))
    last_seen: float = field(default_factory=time.time)
    last_alert_time: float = 0.0
    last_alert_priority: str = ""
    last_risk: float = 0.0
    last_position: str = "ahead"
    last_class: str = ""

TRACK_HISTORY = {}
ALERT_HISTORY = deque(maxlen=200)

# Alert timing settings used by Cell 3.
ALERT_COOLDOWN_SECONDS = 8.0
ALERT_ESCALATION_DELTA = 18.0

# Lower number means higher class priority.
# Critical classes are deliberately ranked above routine detections.
CLASS_PRIORITY = {
    "car": 1,
    "bus": 1,
    "truck": 1,
    "pothole": 1,

    "motorcycle": 2,
    "bicycle": 2,
    "person": 2,

    "obstacle": 3,
    "stairs": 3,
    "crack": 3,
    "manhole": 3,

    "door": 4,

    "traffic_light": 5,
    "traffic_sign": 5,
    "animal": 5,

    "pole": 6,
    "tree": 6,
    "waste_container": 6,
    "bench": 6,
    "chair": 6,
}

# Persistent landmarks may be announced periodically while visible.
PERSISTENT_LANDMARKS = {"door", "stairs"}
LANDMARK_REPEAT_SECONDS = 2.0

# The following state is shared by the FastAPI producer endpoint.
PRODUCER_ACTIVE = False
PRODUCER_ID = None
SERVER_STARTED_AT = time.time()

print("\nClass priority configuration:")
for class_id, class_name in CLASS_NAMES.items():
    normalized_name = str(class_name).strip().lower()
    print(
        f"  {class_id:>2}: {class_name:<18} "
        f"priority={CLASS_PRIORITY.get(normalized_name, 7)}"
    )

print("\nCell 2 completed successfully.")


Loading OptiCore model...
Model loaded.
Device: Tesla T4
FP16 inference: True
Image size: 640
Number of classes: 20
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Warm-up completed in 0.18 seconds

Class priority configuration:
   0: person             priority=2
   1: car                priority=1
   2: bus                priority=1
   3: truck              priority=1
   4: motorcycle         priority=2
   5: bicycle            priority=2
   6: animal             priority=5
   7: door               priority=4
   8: stairs             priority=3
   9: pothole            priority=1
  10: crack              priority=3
  11: manhole            priority=3
  12: traffic_light      priority=5
  13: traffic_sign       priority=5
  14: pole               priority=6
  15: tree               priority=6
  16: bench              priority=6
  17: chair              priority=6
  18: waste_container    priority=6
  19: obstacle           priority=3

Cell 2 

In [39]:

# CELL 3 — OptiCore FastAPI + ByteTrack + priority alerts + WebSocket streaming

import asyncio
import base64
import json
import logging
import time
import uuid
import threading

import cv2
import numpy as np
import torch

from collections import Counter
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import HTMLResponse

logger = logging.getLogger("opticore")
app = FastAPI(title="OptiCore Vision API", version="1.2.0")

# ------------------------------------------------------------
# Configuration
# Uses model/configuration/state initialized in Cell 2.
# ------------------------------------------------------------

MAX_FRAME_BYTES = 8 * 1024 * 1024
MAX_FRAME_WIDTH = 640
JPEG_QUALITY = 65

CRITICAL_CLASSES = {"car", "bus", "truck", "pothole"}
REPEAT_CLASSES = {
    "person", "door", "stairs", "pole", "obstacle",
    "car", "bus", "truck", "pothole",
}
CRITICAL_COOLDOWN = 4.0
REPEAT_COOLDOWN = 4.0

GENERIC_OBSTACLE_CLASSES = {
    "pole", "tree", "bench", "chair",
    "waste_container", "obstacle", "manhole", "crack",
}
APPROACH_CLASSES = {
    "car", "bus", "truck", "motorcycle", "bicycle", "person",
}

# Single persistent YOLO tracker is shared by the server.
# Do not allow multiple simultaneous camera producers.
PRODUCER_ACTIVE = False
PRODUCER_ID = None
SERVER_STARTED_AT = time.time()

# Use the existing locks from Cell 2.
PRODUCER_LOCK = threading.Lock()


def clamp01(value):
    return max(0.0, min(1.0, float(value)))


def infer_position(center_x, width):
    """Camera-image-relative position, not physical world direction."""
    relative_x = float(center_x) / max(1, width)

    if relative_x < 0.38:
        return "left"
    if relative_x > 0.62:
        return "right"
    return "ahead"


def class_priority(name):
    return CLASS_PRIORITY.get(str(name).strip().lower(), 7)


def priority_for_track(track_id, name, box, width, height, now):
    x1, y1, x2, y2 = map(float, box)

    box_w = max(1.0, x2 - x1)
    box_h = max(1.0, y2 - y1)
    center_x = (x1 + x2) / 2.0
    center_y = (y1 + y2) / 2.0
    area = box_w * box_h

    normalized_name = str(name).strip().lower()
    position = infer_position(center_x, width)

    nx = center_x / max(1, width)
    ny = center_y / max(1, height)

    centrality = clamp01(1.0 - abs(nx - 0.5) / 0.5)
    lower_region = clamp01((ny - 0.20) / 0.70)
    path_relevance = clamp01(
        0.65 * centrality + 0.35 * lower_region
    )

    # Update per-track history safely.
    with TRACK_HISTORY_LOCK:
        hist = TRACK_HISTORY.setdefault(
            int(track_id), TrackHistory()
        )

        hist.samples.append((now, center_x, center_y, area))
        hist.last_seen = now
        hist.last_class = normalized_name
        hist.last_position = position

        samples = list(hist.samples)
        prior_risk = hist.last_risk
        last_alert_time = hist.last_alert_time

    motion_score = 0.0
    growth_score = 0.0

    if len(samples) >= 3:
        dt = max(0.05, samples[-1][0] - samples[0][0])

        dx_norm = (
            samples[-1][1] - samples[0][1]
        ) / max(1, width)

        dy_norm = (
            samples[-1][2] - samples[0][2]
        ) / max(1, height)

        motion_score = clamp01(
            ((abs(dx_norm) + abs(dy_norm)) / dt) / 1.2
        )

        old_area = max(1.0, samples[0][3])
        new_area = max(1.0, samples[-1][3])

        growth_per_second = (new_area / old_area - 1.0) / dt
        growth_score = clamp01(growth_per_second / 1.5)

    # Bounding-box growth is only an apparent-approach heuristic.
    approaching = (
        len(samples) >= 3 and growth_score >= 0.16
    )

    if approaching:
        if position == "left":
            approach_direction = "from your left"
        elif position == "right":
            approach_direction = "from your right"
        else:
            approach_direction = "head-on"
    else:
        approach_direction = None

    is_vehicle = normalized_name in {
        "car", "bus", "truck", "motorcycle", "bicycle"
    }

    approach_score = clamp01(
        0.65 * growth_score
        + 0.35 * motion_score * (1.0 if is_vehicle else 0.45)
    )

    persistence = clamp01(len(samples) / 5.0)

    risk_score = 100.0 * (
        0.40 * approach_score
        + 0.30 * path_relevance
        + 0.15 * motion_score
        + 0.15 * persistence
    )

    # Risk ranking is separate from the class priority.
    if (
        risk_score >= 78
        and approach_score >= 0.50
        and path_relevance >= 0.55
    ):
        risk_priority = "P1"
    elif (
        risk_score >= 57
        and (motion_score >= 0.15 or path_relevance >= 0.72)
    ):
        risk_priority = "P2"
    elif risk_score >= 32:
        risk_priority = "P3"
    else:
        risk_priority = "P4"

    cooldown = (
        CRITICAL_COOLDOWN
        if normalized_name in CRITICAL_CLASSES
        else REPEAT_COOLDOWN
        if normalized_name in REPEAT_CLASSES
        else ALERT_COOLDOWN_SECONDS
    )

    # A newly tracked object is eligible immediately.
    first_alert = last_alert_time <= 0.0
    cooldown_elapsed = now - last_alert_time >= cooldown
    material_escalation = risk_score >= prior_risk + ALERT_ESCALATION_DELTA

    # Critical classes get their first alert immediately.
    # Repeats obey cooldown; other classes need escalation after
    # their first alert unless they belong to REPEAT_CLASSES.
    with TRACK_HISTORY_LOCK:
        hist = TRACK_HISTORY[int(track_id)]

        if first_alert:
            alert_candidate = True
        elif cooldown_elapsed and (
            normalized_name in REPEAT_CLASSES or material_escalation
        ):
            alert_candidate = True
        else:
            alert_candidate = False

        hist.last_risk = risk_score

        if alert_candidate:
            hist.last_alert_time = now
            hist.last_alert_priority = risk_priority

    return {
        "position": position,
        "approaching": approaching,
        "approach_direction": approach_direction,
        "class_priority": class_priority(normalized_name),
        "area_ratio": round(area / max(1, width * height), 4),
        "priority": risk_priority,
        "risk_score": round(risk_score, 1),
        "motion_score": round(motion_score, 3),
        "apparent_growth_score": round(growth_score, 3),
        "path_relevance": round(path_relevance, 3),
        "approach_estimate": round(approach_score, 3),
        "alert_candidate": alert_candidate,
        "distance_available": False,
    }


def make_alert_message(detection):
    name = str(detection.get("name", "obstacle")).strip().lower()
    position = detection.get("position", "ahead")

    location = {
        "left": "on your left",
        "right": "on your right",
        "ahead": "ahead",
    }.get(position, "ahead")

    # Prioritize direct, concise warnings for critical classes.
    if name == "pothole":
        return f"Pothole detected {location}."

    if name in {"car", "bus", "truck"}:
        if detection.get("approaching"):
            direction = detection.get("approach_direction")
            if direction == "head-on":
                return f"A {name} appears to be approaching ahead."
            if direction in {"from your left", "from your right"}:
                return f"A {name} appears to be approaching {direction}."
        return f"{name.capitalize()} detected {location}."

    if name in {"motorcycle", "bicycle"}:
        return f"{name.capitalize()} detected {location}."

    if name == "person":
        return f"Person detected {location}."

    if name == "stairs":
        return f"Stairs detected {location}."

    if name == "door":
        return f"Door detected {location}."

    if name in GENERIC_OBSTACLE_CLASSES:
        return f"{name.replace('_', ' ').capitalize()} detected {location}."

    spoken_name = name.replace("_", " ")
    article = "an" if spoken_name[:1] in "aeiou" else "a"
    return f"{article.capitalize()} {spoken_name} detected {location}."


def process_frame(frame):
    start = time.perf_counter()
    now = time.time()

    # Lock prevents concurrent calls from corrupting shared ByteTrack state.
    with INFERENCE_LOCK, torch.inference_mode():
        results = model.track(
            source=frame,
            persist=True,
            tracker="bytetrack.yaml",
            imgsz=IMGSZ,
            conf=CONFIDENCE,
            device=DEVICE,
            verbose=False,
        )

        result = results[0]
        annotated = result.plot()

        detections = []
        frame_counts = Counter()

        height, width = frame.shape[:2]
        boxes = result.boxes

        if boxes is not None and len(boxes):
            coords = boxes.xyxy.cpu().numpy()
            scores = boxes.conf.cpu().numpy()
            classes = boxes.cls.cpu().numpy().astype(int)

            if boxes.id is not None:
                track_ids = boxes.id.int().cpu().tolist()
            else:
                track_ids = [None] * len(coords)

            for box, score, cls_id, track_id in zip(
                coords, scores, classes, track_ids
            ):
                cls_id = int(cls_id)
                name = str(CLASS_NAMES[cls_id])
                normalized_name = name.strip().lower()

                x1, y1, x2, y2 = map(float, box)
                frame_counts[name] += 1

                if track_id is not None:
                    UNIQUE_TRACKS[name].add(int(track_id))

                detection = {
                    "name": name,
                    "confidence": round(float(score), 3),
                    "id": int(track_id) if track_id is not None else None,
                    "position": infer_position((x1 + x2) / 2, width),
                    "class_priority": class_priority(normalized_name),
                    "area_ratio": round(
                        max(0.0, x2 - x1)
                        * max(0.0, y2 - y1)
                        / max(1, width * height),
                        4,
                    ),
                    "priority": "P4",
                    "risk_score": 0.0,
                    "approaching": False,
                    "approach_direction": None,
                    "alert_candidate": False,
                    "distance_available": False,
                }

                # Persistent per-track alerts require a ByteTrack ID.
                # Untracked detections are shown but not repeatedly spoken.
                if track_id is not None:
                    detection.update(
                        priority_for_track(
                            int(track_id),
                            name,
                            box,
                            width,
                            height,
                            now,
                        )
                    )

                detections.append(detection)

        with METRICS_LOCK:
            CURRENT_COUNTS.clear()
            CURRENT_COUNTS.update(frame_counts)

    elapsed_ms = (time.perf_counter() - start) * 1000.0

    with METRICS_LOCK:
        METRICS["frames_processed"] += 1
        METRICS["last_inference_ms"] = round(elapsed_ms, 2)

        previous = METRICS["avg_inference_ms"]
        METRICS["avg_inference_ms"] = round(
            elapsed_ms if previous == 0
            else previous * 0.9 + elapsed_ms * 0.1,
            2,
        )

        METRICS["last_inference_unix"] = now

        instantaneous_fps = 1000.0 / max(1.0, elapsed_ms)
        old_fps = METRICS["fps_ema"]
        METRICS["fps_ema"] = round(
            instantaneous_fps if old_fps == 0
            else old_fps * 0.9 + instantaneous_fps * 0.1,
            2,
        )

        metrics_counts = dict(CURRENT_COUNTS)
        unique_counts = {
            name: len(ids) for name, ids in UNIQUE_TRACKS.items()
        }

    # Highest class priority first, then higher confidence.
    detections.sort(
        key=lambda d: (
            int(d.get("class_priority", 7)),
            -float(d.get("confidence", 0.0)),
        )
    )

    counts = {
        "current": metrics_counts,
        "unique_tracked": unique_counts,
    }

    alerts = []

    # Put urgent alerts first in each response.
    alert_detections = sorted(
        (
            d for d in detections
            if d.get("alert_candidate")
        ),
        key=lambda d: (
            int(d.get("class_priority", 7)),
            -float(d.get("risk_score", 0.0)),
        ),
    )

    for detection in alert_detections:
        alert = {
            "timestamp": now,
            "track_id": detection.get("id"),
            "name": str(detection.get("name", "")).lower(),
            "confidence": detection.get("confidence", 0.0),
            "class_priority": detection.get("class_priority", 7),
            "position": detection.get("position", "ahead"),
            "approaching": detection.get("approaching", False),
            "approach_direction": detection.get("approach_direction"),
            "priority": detection.get("priority", "P4"),
            "risk_score": detection.get("risk_score", 0),
            "message": make_alert_message(detection),
        }

        alerts.append(alert)

        with ALERT_HISTORY_LOCK:
            ALERT_HISTORY.appendleft(alert)

    return annotated, detections, elapsed_ms, counts, alerts


# ------------------------------------------------------------
# HTTP monitoring endpoints
# ------------------------------------------------------------

@app.get("/")
async def browser_page():
    return HTMLResponse(BROWSER_HTML)


@app.get("/health")
async def health():
    with METRICS_LOCK:
        metrics = dict(METRICS)

    return {
        "status": "ok",
        "model_loaded": model is not None,
        "device": str(DEVICE),
        "classes": len(CLASS_NAMES),
        "tracker": "bytetrack.yaml",
        "imgsz": IMGSZ,
        **metrics,
    }


@app.get("/status")
async def status():
    with METRICS_LOCK:
        metrics = dict(METRICS)

    return {
        "service": "OptiCore Vision API",
        "version": "1.2.0",
        "model_loaded": model is not None,
        "device": str(DEVICE),
        "gpu_available": bool(torch.cuda.is_available()),
        "gpu_name": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available() else None
        ),
        "producer_active": PRODUCER_ACTIVE,
        "producer_id": PRODUCER_ID,
        "uptime_seconds": round(time.time() - SERVER_STARTED_AT, 1),
        "metrics": metrics,
        "priority_engine": "class priority + image-only risk heuristic",
        "speech_policy": "browser speech + track-level server deduplication",
    }


@app.get("/metrics")
async def metrics():
    with METRICS_LOCK:
        return dict(METRICS)


@app.get("/events")
async def events(limit: int = 20):
    limit = max(1, min(100, int(limit)))

    with ALERT_HISTORY_LOCK:
        recent = list(ALERT_HISTORY)[:limit]

    return {"events": recent}


# ------------------------------------------------------------
# Real-time browser camera WebSocket
# Protocol preserved for Cell 4:
# client sends binary JPEG; server returns JSON text containing
# base64 annotated JPEG, detections, counts and alerts.
# ------------------------------------------------------------

@app.websocket("/ws/video")
async def video_stream(websocket: WebSocket):
    global PRODUCER_ACTIVE, PRODUCER_ID

    await websocket.accept()

    # The tracker state is global, so only one camera producer is allowed.
    with PRODUCER_LOCK:
        if PRODUCER_ACTIVE:
            await websocket.send_text(json.dumps({
                "type": "error",
                "code": "producer_busy",
                "message": (
                    "Another camera stream is active. "
                    "Stop it before connecting another camera."
                ),
            }))
            await websocket.close(code=1013)
            return

        PRODUCER_ACTIVE = True
        PRODUCER_ID = str(uuid.uuid4())[:8]
        this_producer = PRODUCER_ID

    logger.info("Camera producer connected: %s", this_producer)

    try:
        while True:
            payload = await websocket.receive_bytes()

            if not payload or len(payload) > MAX_FRAME_BYTES:
                with METRICS_LOCK:
                    METRICS["frames_rejected"] += 1
                continue

            frame = cv2.imdecode(
                np.frombuffer(payload, dtype=np.uint8),
                cv2.IMREAD_COLOR,
            )

            if frame is None:
                with METRICS_LOCK:
                    METRICS["frames_rejected"] += 1
                continue

            height, width = frame.shape[:2]

            if width > MAX_FRAME_WIDTH:
                new_height = max(
                    1, int(height * MAX_FRAME_WIDTH / width)
                )
                frame = cv2.resize(
                    frame,
                    (MAX_FRAME_WIDTH, new_height),
                    interpolation=cv2.INTER_AREA,
                )

            (
                annotated,
                detections,
                inference_ms,
                counts,
                alerts,
            ) = await asyncio.to_thread(process_frame, frame)

            ok, encoded = cv2.imencode(
                ".jpg",
                annotated,
                [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY],
            )

            if not ok:
                continue

            response = {
                "image": base64.b64encode(
                    encoded.tobytes()
                ).decode("ascii"),
                "detections": detections,
                "counts": counts,
                "alerts": alerts,
                "inference_ms": round(inference_ms, 2),
                "stream_id": this_producer,
                "timestamp": time.time(),
            }

            await websocket.send_text(
                json.dumps(response, separators=(",", ":"))
            )

    except WebSocketDisconnect:
        logger.info(
            "Camera client disconnected: %s", this_producer
        )

    except Exception:
        logger.exception("OptiCore WebSocket processing failed")

        try:
            await websocket.close(code=1011)
        except Exception:
            pass

    finally:
        with PRODUCER_LOCK:
            if PRODUCER_ID == this_producer:
                PRODUCER_ACTIVE = False
                PRODUCER_ID = None

        # Remove stale track history to keep memory bounded.
        cutoff = time.time() - 30.0

        with TRACK_HISTORY_LOCK:
            stale_ids = [
                track_id
                for track_id, history in TRACK_HISTORY.items()
                if history.last_seen < cutoff
            ]

            for track_id in stale_ids:
                del TRACK_HISTORY[track_id]

print("OptiCore FastAPI application created.")
print("Routes:", [
    getattr(route, "path", "?")
    for route in app.routes
])
print("Priority classes:", sorted(CRITICAL_CLASSES))
print("Streaming endpoint: /ws/video")
print("Monitoring endpoints: /health, /status, /metrics, /events")
print("Cell 3 completed successfully.")


OptiCore FastAPI application created.
Routes: ['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc', '/', '/health', '/status', '/metrics', '/events', '/ws/video']
Priority classes: ['bus', 'car', 'pothole', 'truck']
Streaming endpoint: /ws/video
Monitoring endpoints: /health, /status, /metrics, /events
Cell 3 completed successfully.


In [40]:

# CELL 4 — Native OptiCore browser UI; no Gradio

BROWSER_HTML = r"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>OptiCore | Live Vision</title>
<style>
:root {
  color-scheme: dark;
  --bg: #080d16;
  --panel: #111a29;
  --border: #26364b;
  --text: #f1f5f9;
  --muted: #a8b7cb;
  --blue: #60a5fa;
  --red: #ff6464;
  --yellow: #facc15;
}
* { box-sizing: border-box; }
body {
  margin: 0;
  background: var(--bg);
  color: var(--text);
  font: 15px/1.5 system-ui, sans-serif;
}
header {
  padding: 16px 20px;
  background: #0d1523;
  border-bottom: 1px solid var(--border);
  display: flex;
  align-items: center;
  justify-content: space-between;
  flex-wrap: wrap;
  gap: 10px;
}
h1 { font-size: 22px; margin: 0; letter-spacing: .3px; }
.subtitle { color: var(--muted); font-size: 13px; }
main {
  max-width: 1300px;
  margin: auto;
  padding: 18px;
  display: grid;
  grid-template-columns: minmax(0, 1.7fr) minmax(280px, 1fr);
  gap: 16px;
}
.panel {
  min-width: 0;
  background: var(--panel);
  border: 1px solid var(--border);
  border-radius: 14px;
  padding: 15px;
}
h2 { margin: 0 0 12px; font-size: 17px; }
video, canvas#output {
  display: block;
  width: 100%;
  background: #03060b;
  border-radius: 9px;
  aspect-ratio: 4 / 3;
  object-fit: contain;
}
video { display: none; }
.controls { display: flex; flex-wrap: wrap; gap: 9px; margin-top: 12px; }
button, select {
  color: var(--text);
  background: #1c2b40;
  border: 1px solid #3a506d;
  border-radius: 8px;
  padding: 10px 13px;
  font: inherit;
}
button { cursor: pointer; }
button.primary { background: #1d4ed8; border-color: #3b82f6; }
button:disabled { opacity: .45; cursor: not-allowed; }
.status {
  margin-top: 12px;
  padding: 10px;
  border-radius: 8px;
  background: #0b1220;
  color: var(--muted);
  overflow-wrap: anywhere;
}
.metrics {
  display: grid;
  grid-template-columns: repeat(2, minmax(0, 1fr));
  gap: 9px;
  margin: 12px 0;
}
.metric {
  padding: 11px;
  background: #0b1220;
  border: 1px solid var(--border);
  border-radius: 9px;
}
.metric span { color: var(--muted); font-size: 12px; display: block; }
.metric strong { font-size: 20px; }
ul { list-style: none; padding: 0; margin: 0; }
li {
  padding: 9px 0;
  border-bottom: 1px solid var(--border);
  overflow-wrap: anywhere;
}
.priority { color: var(--red); font-weight: 750; }
.normal { color: var(--blue); }
#alerts { max-height: 250px; overflow: auto; }
#detections { max-height: 370px; overflow: auto; }
.small { color: var(--muted); font-size: 12px; }
.badge {
  display: inline-block;
  border: 1px solid var(--border);
  padding: 2px 7px;
  border-radius: 5px;
  margin-right: 5px;
  font-size: 11px;
}
.header-right {
  display: flex;
  align-items: center;
  gap: 12px;
}
.opticore-logo {
  width: 52px;
  height: 52px;
  object-fit: contain;
  border-radius: 9px;
  flex-shrink: 0;
}
@media(max-width: 850px) {
  main { grid-template-columns: 1fr; padding: 10px; }
  header { padding: 13px; }
}
</style>
</head>
<body>
<header>
  <div>
    <h1>OptiCore</h1>
    <div class="subtitle">Real-time assistive vision · YOLO + ByteTrack</div>
  </div>
  <div class="header-right">
    <div id="connection" class="badge">Disconnected</div>
    <img class="opticore-logo"
      src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQnmI058l9sC8o6AfTDISbN1ZrDjfSH66wyAoy_9lq6uQ&s"
      alt="OptiCore logo"
      referrerpolicy="no-referrer">
  </div>
</header>

<main>
  <section class="panel">
    <h2>Live annotated camera</h2>
    <canvas id="output" width="640" height="480"></canvas>
    <video id="source" autoplay muted playsinline></video>

    <div class="controls">
      <button id="start" class="primary">Start camera</button>
      <button id="stop" disabled>Stop camera</button>
      <button id="voice">Enable voice</button>
      <select id="cameraFacing" aria-label="Camera">
        <option value="environment">Rear camera</option>
        <option value="user">Front camera</option>
      </select>
    </div>

    <div id="status" class="status">Ready. Start the camera to begin.</div>

    <div class="metrics">
      <div class="metric">
        <span>Inference time</span>
        <strong id="latency">—</strong>
      </div>
      <div class="metric">
        <span>Detections / frame</span>
        <strong id="detCount">0</strong>
      </div>
      <div class="metric">
        <span>Processed frames</span>
        <strong id="frameCount">0</strong>
      </div>
      <div class="metric">
        <span>Stream rate</span>
        <strong id="fps">—</strong>
      </div>
    </div>
    <div class="small">
      High-priority classes: car, bus, truck and pothole.
      Voice output is generated locally by your browser.
    </div>
  </section>

  <section>
    <div class="panel">
      <h2>Priority alerts</h2>
      <ul id="alerts"><li class="small">No alerts yet.</li></ul>
    </div>

    <div class="panel" style="margin-top:16px">
      <h2>Current detections</h2>
      <ul id="detections"><li class="small">Waiting for camera.</li></ul>
    </div>
  </section>
</main>

<script>
(() => {
  const $ = id => document.getElementById(id);
  const video = $("source");
  const canvas = $("output");
  const ctx = canvas.getContext("2d", { alpha: false });

  let stream = null;
  let ws = null;
  let active = false;
  let voiceEnabled = false;
  let waitingForResponse = false;
  let lastFrameAt = 0;
  let frameCount = 0;
  let previousFrameCountAt = performance.now();
  let fpsEstimate = 0;
  let frameTimer = null;
  let reconnectTimer = null;
  let spokenRecently = new Map();

  const HIGH = new Set(["car", "bus", "truck", "pothole"]);

  function setStatus(message) {
    $("status").textContent = message;
  }

  function setConnection(message) {
    $("connection").textContent = message;
  }

  function websocketURL() {
    const scheme = location.protocol === "https:" ? "wss:" : "ws:";
    return `${scheme}//${location.host}/ws/video`;
  }

  // Batch detections so a busy road does not produce overlapping announcements.
  const pendingVoiceAlerts = new Map();
  let voiceBatchTimer = null;
  let voiceSpeaking = false;
  const voiceQueue = [];
  let lastVoiceMessage = "";
  let lastVoiceMessageAt = 0;
  const VOICE_BATCH_MS = 1600;
  const VOICE_REPEAT_COOLDOWN_MS = 7000;

  function speakAlert(alert) {
    if (!voiceEnabled || !("speechSynthesis" in window)) return;
    if (!alert || !alert.message) return;

    const name = String(alert.name || "").toLowerCase();
    let group = name;
    if (["car", "bus", "truck", "motorcycle", "bicycle"].includes(name)) {
      group = "vehicles";
    } else if (["person"].includes(name)) {
      group = "people";
    }

    // Keep one representative event per group during each short batch window.
    if (!pendingVoiceAlerts.has(group) || HIGH.has(name)) {
      pendingVoiceAlerts.set(group, alert);
    }

    if (!voiceBatchTimer) {
      voiceBatchTimer = setTimeout(flushVoiceAlerts, VOICE_BATCH_MS);
    }
  }

  function flushVoiceAlerts() {
    voiceBatchTimer = null;
    if (!pendingVoiceAlerts.size || !voiceEnabled) {
      pendingVoiceAlerts.clear();
      return;
    }

    const groups = [...pendingVoiceAlerts.keys()];
    const alerts = [...pendingVoiceAlerts.values()];
    pendingVoiceAlerts.clear();

    const hasVehicles = groups.includes("vehicles");
    const hasPeople = groups.includes("people");
    const hasPothole = alerts.some(a => String(a.name || "").toLowerCase() === "pothole");
    const other = alerts.filter(a => !["vehicles", "people"].includes(
      ["car", "bus", "truck", "motorcycle", "bicycle"].includes(String(a.name || "").toLowerCase())
        ? "vehicles"
        : String(a.name || "").toLowerCase() === "person" ? "people" : String(a.name || "").toLowerCase()
    ));

    let message;
    if (hasVehicles && hasPothole) {
      message = "Vehicles detected. A pothole is also detected; watch your path.";
    } else if (hasVehicles && hasPeople) {
      message = "Vehicles and people detected nearby. Stay aware.";
    } else if (hasVehicles) {
      message = "Vehicles detected nearby. Stay aware.";
    } else if (hasPeople) {
      message = "People detected nearby.";
    } else if (alerts.length) {
      // Keep non-vehicle messages concise and avoid repeating every frame.
      message = alerts[0].message;
    } else {
      return;
    }

    const now = Date.now();
    if (message === lastVoiceMessage &&
        now - lastVoiceMessageAt < VOICE_REPEAT_COOLDOWN_MS) return;

    lastVoiceMessage = message;
    lastVoiceMessageAt = now;
    voiceQueue.push(message);
    processVoiceQueue();
  }

  function processVoiceQueue() {
    if (voiceSpeaking || !voiceQueue.length || !voiceEnabled) return;
    if (!("speechSynthesis" in window)) return;

    const message = voiceQueue.shift();
    const utterance = new SpeechSynthesisUtterance(message);
    utterance.lang = "en-IN";
    utterance.rate = 0.92;
    utterance.pitch = 1.0;
    utterance.volume = 1.0;
    voiceSpeaking = true;

    utterance.onend = () => {
      voiceSpeaking = false;
      processVoiceQueue();
    };
    utterance.onerror = () => {
      voiceSpeaking = false;
      processVoiceQueue();
    };

    // Do not cancel current speech when another vehicle appears.
    window.speechSynthesis.speak(utterance);
  }

  function renderAlerts(alerts) {
    if (!Array.isArray(alerts) || !alerts.length) return;

    const list = $("alerts");
    if (list.children.length === 1 &&
        list.textContent.includes("No alerts yet")) {
      list.innerHTML = "";
    }

    for (const alert of alerts) {
      const li = document.createElement("li");
      const critical = HIGH.has(String(alert.name || "").toLowerCase());
      li.className = critical ? "priority" : "normal";

      const badge = document.createElement("span");
      badge.className = "badge";
      badge.textContent = critical ? "HIGH" : (alert.priority || "INFO");

      const message = document.createElement("span");
      message.textContent = alert.message || "Detection event";

      li.append(badge, message);
      list.prepend(li);
      speakAlert(alert);
    }

    while (list.children.length > 30) list.lastElementChild.remove();
  }

  function renderDetections(detections) {
    const list = $("detections");
    list.innerHTML = "";

    if (!detections || !detections.length) {
      list.innerHTML = '<li class="small">No objects detected in this frame.</li>';
      return;
    }

    for (const d of detections) {
      const li = document.createElement("li");
      const name = String(d.name || "object");
      const critical = HIGH.has(name.toLowerCase());
      const id = d.id === null || d.id === undefined ? "untracked" : `#${d.id}`;
      const confidence = Math.round((Number(d.confidence) || 0) * 100);

      const badge = document.createElement("span");
      badge.className = "badge";
      badge.textContent = critical ? "HIGH" : (d.priority || "OBJECT");

      const text = document.createElement("span");
      text.textContent = `${name} ${id} · ${confidence}% · ${d.position || "ahead"}`;

      li.className = critical ? "priority" : "";
      li.append(badge, text);
      list.appendChild(li);
    }
  }

  function drawPlaceholder(message) {
    canvas.width = 640;
    canvas.height = 480;
    ctx.fillStyle = "#03060b";
    ctx.fillRect(0, 0, canvas.width, canvas.height);
    ctx.fillStyle = "#a8b7cb";
    ctx.font = "16px system-ui";
    ctx.textAlign = "center";
    ctx.fillText(message, canvas.width / 2, canvas.height / 2);
  }

  function connectSocket() {
    if (!active) return;

    if (ws && (ws.readyState === WebSocket.OPEN ||
               ws.readyState === WebSocket.CONNECTING)) return;

    setConnection("Connecting…");
    ws = new WebSocket(websocketURL());
    ws.binaryType = "arraybuffer";

    ws.onopen = () => {
      if (!active) {
        ws.close();
        return;
      }
      setConnection("Connected");
      setStatus("Connected to OptiCore inference server.");
      sendNextFrame();
    };

    ws.onmessage = async event => {
      try {
        if (typeof event.data !== "string") {
          throw new Error("Unexpected binary response from server.");
        }

        const data = JSON.parse(event.data);

        if (data.type === "error") {
          setStatus(data.message || "Server rejected the stream.");
          active = false;
          $("start").disabled = false;
          $("stop").disabled = true;
          return;
        }

        if (data.image) {
          const image = new Image();
          image.src = "data:image/jpeg;base64," + data.image;

          await new Promise((resolve, reject) => {
            image.onload = resolve;
            image.onerror = reject;
          });

          canvas.width = image.naturalWidth;
          canvas.height = image.naturalHeight;
          ctx.drawImage(image, 0, 0);
        }

        const detections = data.detections || [];
        const alerts = data.alerts || [];

        $("latency").textContent =
          Number.isFinite(Number(data.inference_ms))
            ? `${Number(data.inference_ms).toFixed(1)} ms`
            : "—";

        $("detCount").textContent = detections.length;
        frameCount++;
        $("frameCount").textContent = frameCount;

        const now = performance.now();
        const delta = now - previousFrameCountAt;
        if (delta > 0) {
          const instantaneous = 1000 / delta;
          fpsEstimate = fpsEstimate
            ? fpsEstimate * 0.8 + instantaneous * 0.2
            : instantaneous;
        }
        previousFrameCountAt = now;
        $("fps").textContent = `${fpsEstimate.toFixed(1)} FPS`;

        renderDetections(detections);
        renderAlerts(alerts);

        setStatus(
          `Frame processed · ${detections.length} detections · ` +
          `${Number(data.inference_ms || 0).toFixed(1)} ms inference`
        );
      } catch (err) {
        setStatus(`Response error: ${err.message}`);
      } finally {
        waitingForResponse = false;
        if (active && ws && ws.readyState === WebSocket.OPEN) {
          sendNextFrame();
        }
      }
    };

    ws.onerror = () => {
      setConnection("Connection error");
    };

    ws.onclose = () => {
      setConnection("Disconnected");
      waitingForResponse = false;

      if (active) {
        setStatus("Connection lost. Reconnecting…");
        clearTimeout(reconnectTimer);
        reconnectTimer = setTimeout(connectSocket, 1500);
      }
    };
  }

  function sendNextFrame() {
    if (!active || !stream || !ws ||
        ws.readyState !== WebSocket.OPEN || waitingForResponse) return;

    // Throttle capture and keep at most one request in flight.
    const now = performance.now();
    const minInterval = 150; // Maximum target ~6.7 FPS.
    const elapsed = now - lastFrameAt;

    if (elapsed < minInterval) {
      clearTimeout(frameTimer);
      frameTimer = setTimeout(sendNextFrame, minInterval - elapsed);
      return;
    }

    if (video.readyState < HTMLMediaElement.HAVE_CURRENT_DATA) {
      frameTimer = setTimeout(sendNextFrame, 100);
      return;
    }

    lastFrameAt = now;

    const maxWidth = 640;
    const scale = Math.min(1, maxWidth / video.videoWidth);
    const width = Math.max(1, Math.round(video.videoWidth * scale));
    const height = Math.max(1, Math.round(video.videoHeight * scale));

    const capture = document.createElement("canvas");
    capture.width = width;
    capture.height = height;
    capture.getContext("2d", { alpha: false })
      .drawImage(video, 0, 0, width, height);

    waitingForResponse = true;

    capture.toBlob(blob => {
      if (!active || !blob || !ws ||
          ws.readyState !== WebSocket.OPEN) {
        waitingForResponse = false;
        return;
      }
      blob.arrayBuffer().then(buffer => {
        if (active && ws && ws.readyState === WebSocket.OPEN) {
          ws.send(buffer);
        } else {
          waitingForResponse = false;
        }
      }).catch(err => {
        waitingForResponse = false;
        setStatus(`Frame encoding error: ${err.message}`);
      });
    }, "image/jpeg", 0.65);
  }

  async function startCamera() {
    if (active) return;

    try {
      if (!navigator.mediaDevices ||
          !navigator.mediaDevices.getUserMedia) {
        throw new Error("Camera access requires HTTPS or localhost.");
      }

      $("start").disabled = true;
      setStatus("Requesting camera permission…");

      stream = await navigator.mediaDevices.getUserMedia({
        audio: false,
        video: {
          facingMode: { ideal: $("cameraFacing").value },
          width: { ideal: 640 },
          height: { ideal: 480 },
          frameRate: { ideal: 15, max: 20 }
        }
      });

      video.srcObject = stream;
      await video.play();

      active = true;
      frameCount = 0;
      $("frameCount").textContent = "0";
      $("stop").disabled = false;
      connectSocket();
    } catch (err) {
      active = false;
      $("start").disabled = false;
      $("stop").disabled = true;
      setStatus(`Camera error: ${err.message}`);
    }
  }

  function stopCamera() {
    active = false;
    waitingForResponse = false;

    clearTimeout(frameTimer);
    clearTimeout(reconnectTimer);

    if (ws) {
      ws.onclose = null;
      ws.close();
      ws = null;
    }

    if (stream) {
      stream.getTracks().forEach(track => track.stop());
      stream = null;
    }

    video.srcObject = null;
    $("start").disabled = false;
    $("stop").disabled = true;
    setConnection("Disconnected");
    setStatus("Camera stopped.");

    if ("speechSynthesis" in window) {
      window.speechSynthesis.cancel();
    }
  }

  $("start").addEventListener("click", startCamera);
  $("stop").addEventListener("click", stopCamera);

  $("voice").addEventListener("click", () => {
    voiceEnabled = !voiceEnabled;
    $("voice").textContent = voiceEnabled ? "Disable voice" : "Enable voice";

    if (voiceEnabled && "speechSynthesis" in window) {
      // User gesture unlocks browser speech on supported browsers.
      window.speechSynthesis.cancel();
      const utterance = new SpeechSynthesisUtterance("OptiCore voice alerts enabled.");
      utterance.rate = 1.0;
      window.speechSynthesis.speak(utterance);
    } else if ("speechSynthesis" in window) {
      window.speechSynthesis.cancel();
    }
  });

  window.addEventListener("beforeunload", stopCamera);
  drawPlaceholder("Start camera to begin OptiCore");
})();
</script>
</body>
</html>
"""

assert isinstance(BROWSER_HTML, str) and "<html" in BROWSER_HTML.lower()
assert "/ws/video" in BROWSER_HTML
assert "gradio" not in BROWSER_HTML.lower()

print("Native OptiCore browser UI created.")
print("Gradio: not used")
print("Camera: browser getUserMedia")
print("Streaming: persistent WebSocket with one frame in flight")
print("Voice: browser speech synthesis with critical-class priority")
print("Cell 4 completed successfully.")


Native OptiCore browser UI created.
Gradio: not used
Camera: browser getUserMedia
Streaming: persistent WebSocket with one frame in flight
Voice: browser speech synthesis with critical-class priority
Cell 4 completed successfully.


In [41]:

# CELL 5 — Start OptiCore API and expose it through ngrok

import os
import time
import socket
import threading
import requests
import uvicorn

from pyngrok import ngrok
from IPython.display import display, HTML

PORT = 8000
HOST = "0.0.0.0"

# ------------------------------------------------------------
# 1. Retrieve ngrok token securely from Kaggle Secrets
# ------------------------------------------------------------

NGROK_TOKEN = None

try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    NGROK_TOKEN = secrets.get_secret("NGROK_AUTHTOKEN")
except Exception as exc:
    print("Could not read Kaggle secret:", str(exc))

# Environment variable is a fallback for environments outside Kaggle.
if not NGROK_TOKEN:
    NGROK_TOKEN = os.environ.get("NGROK_AUTHTOKEN")

if not NGROK_TOKEN:
    raise RuntimeError(
        "Missing NGROK_AUTHTOKEN. Add it in Kaggle Secrets, "
        "enable notebook access, then rerun Cell 5."
    )

# ------------------------------------------------------------
# 2. Configure ngrok and start FastAPI
# ------------------------------------------------------------

# Avoid accidentally opening multiple tunnels on repeated execution.
try:
    for tunnel in ngrok.get_tunnels():
        if tunnel.config.get("addr") in (str(PORT), f"http://localhost:{PORT}"):
            ngrok.disconnect(tunnel.public_url)
except Exception:
    pass

# Reuse the existing ngrok configuration if the token is already saved.
try:
    ngrok.set_auth_token(NGROK_TOKEN)
except Exception as e:
    if "Authtoken already set in configuration file" in str(e):
        print("Using the existing ngrok authentication configuration.")
    else:
        raise

# Bind the server in a background thread so the notebook stays usable.
if "opticore_uvicorn_server" in globals():
    try:
        if opticore_uvicorn_server.started:
            print("An OptiCore server is already running.")
    except Exception:
        pass

config = uvicorn.Config(
    app=app,
    host=HOST,
    port=PORT,
    log_level="info",
    access_log=False,
)

opticore_uvicorn_server = uvicorn.Server(config)

def run_opticore_server():
    opticore_uvicorn_server.run()

server_thread = threading.Thread(
    target=run_opticore_server,
    name="opticore-api",
    daemon=True,
)

server_thread.start()

# Wait for local API readiness before creating the tunnel.
local_url = f"http://127.0.0.1:{PORT}"
deadline = time.time() + 60

while time.time() < deadline:
    try:
        response = requests.get(
            f"{local_url}/health",
            timeout=2,
        )
        if response.ok:
            break
    except requests.RequestException:
        pass

    if not server_thread.is_alive():
        raise RuntimeError(
            "FastAPI exited before becoming ready. "
            "Review the Uvicorn logs above."
        )

    time.sleep(1)
else:
    raise TimeoutError(
        "FastAPI did not become ready within 60 seconds."
    )

# Create a public HTTPS tunnel to the local API.
tunnel = ngrok.connect(PORT, "http")
PUBLIC_URL = tunnel.public_url.rstrip("/")

# Prefer HTTPS so mobile browsers can request camera permission.
if PUBLIC_URL.startswith("http://"):
    PUBLIC_URL = "https://" + PUBLIC_URL[len("http://"):]

# ------------------------------------------------------------
# 3. Verify the public endpoint
# ------------------------------------------------------------

health_response = requests.get(
    f"{PUBLIC_URL}/health",
    timeout=20,
)

print("\n" + "=" * 55)
print("OPTICORE SERVER STATUS")
print("=" * 55)
print("Local API:", local_url)
print("Public URL:", PUBLIC_URL)
print("Browser UI:", PUBLIC_URL + "/")
print("Health:", health_response.status_code)
print("Health response:", health_response.json())
print("\nOpen the Browser UI URL in your device.")
print("Click Start camera, then Enable voice.")
print("Keep this Kaggle notebook running while using OptiCore.")


INFO:pyngrok.process:Updating authtoken for default "config_path" of "ngrok_path": /root/.config/ngrok/ngrok
INFO:     Started server process [48]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:pyngrok.ngrok:Opening tunnel named: http-8000-3560feee-81f7-4b79-8491-addf258a0fe0
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Using the existing ngrok authentication configuration.

OPTICORE SERVER STATUS
Local API: http://127.0.0.1:8000
Public URL: https://distill-shrouded-washing.ngrok-free.dev
Browser UI: https://distill-shrouded-washing.ngrok-free.dev/
Health: 200
Health response: {'status': 'ok', 'model_loaded': True, 'device': '0', 'classes': 20, 'tracker': 'bytetrack.yaml', 'imgsz': 640, 'frames_processed': 45, 'last_inference_ms': 15.58, 'avg_inference_ms': 17.24, 'fps_ema': 63.09, 'frames_rejected': 0, 'last_inference_unix': 1793427704.9710028}

Open the Browser UI URL in your device.
Click Start camera, then Enable voice.
Keep this Kaggle notebook running while using OptiCore.


INFO:     2406:7400:10c:cc88:e0e2:8415:436e:748:0 - "WebSocket /ws/video" [accepted]
INFO:     connection open
INFO:opticore:Camera producer connected: 189b8cdf
INFO:opticore:OptiCore client disconnected: 189b8cdf
INFO:opticore:OptiCore client disconnected: 3b0fbb89
